# Pendeteksi Berita Palsu Menggunakan Model Artificial Neural Network dengan Pytorch

## Persiapan

### Import Library yang Digunakan
Pada proses ini dilakukan import library yang akan digunakan

In [ ]:
#Data Manupulasi
import numpy as np
import pandas as pd

#Natural Language Processing
import re 
import nltk 
from sklearn.feature_extraction.text import CountVectorizer

#Pytorch
import torch
import torch.nn as nn

#Visualisasi
import matplotlib.pyplot as plt
import seaborn as sns

#Training, Testing
from sklearn.model_selection import train_test_split

### Import Dataset yang Digunakan
Selanjutnya import dataset yang disediakan. Pada tahap ini menggunakan dua dataset yang berisikan contoh berita asli dan palsu. Setelah diimport, lihat informasi tentang struktur dan tipe data dalam dataset.

In [ ]:
#Membaca Dataset

asli = pd.read_csv("True.csv")
palsu = pd.read_csv("Fake.csv")

In [ ]:
#Mendapatkan informasi tentang struktur dan tipe data dalam Dataset 'asli'

asli.info()

In [ ]:
##Mendapatkan informasi tentang struktur dan tipe data dalam Dataset 'palsu'

palsu.info()

### Penambahan Label Untuk Masing-Masing Dataset
Jika sudah berhasil diimport, langkah selanjutnya adalah memberikan label pada setiap dataset. Label digunakan untuk membedakan dan memberi nilai antara berita palsu dengan berita asli. Berita palsu diberi nilai 0 dan berita asli diberi nilai 1. Nilai pada label menggunakan type integer.

In [ ]:
#menambahkan kolom "label" pada Data asli dan palsi dengan nilai yang menandakan apakah berita tersebut dikategorikan sebagai benar atau palsu.

asli["label"] = np.ones(len(asli),dtype=int)
palsu["label"] = np.zeros(len(palsu),dtype=int)

In [ ]:
#menampilkan beberapa baris pertama dari Data 'asli'

asli.head()

In [ ]:
#menampilkan beberapa baris pertama dari Data 'fake'

palsu.head()

### Penggabungan Dataset
Setelah diberi label, kedua dataset tersebut digabungkan menjadi satu dataset degan menggunakan fungsi pd.concat() dan hasilnya disimpan dalam variabel data.

In [ ]:
#menggabungkan dua Dataset, 'asli' dan 'palsu', menjadi satu Data tunggal dengan menggunakan fungsi pd.concat().
#hasil penggabungan disimpan dalam variabel data.
#data.info() untuk mencetak informasi tentang struktur dan tipe data dalam DataFrame 'data'.

data = pd.concat((asli,palsu),axis=0)
print(data.info())

### Pengacakan Dataset
Setelah dataset digabungkan, dilakukan pengacakan baris dalam dataset agar tidak terjadi bias saat melakukan training.

In [ ]:
#data = data.sample(frac=1): Baris ini digunakan untuk mengacak baris-baris dalam DataFrame data.
#fungsi sample() digunakan untuk mengambil sampel acak dari DataFrame. 
#frac=1 mengambil seluruh DataFrame dengan urutan acak (frac=1 berarti mengambil 100% dari data). Dengan mengacak baris-baris dan mengubah urutan data dalam DataFrame secara acak.

data = data.sample(frac=1)
data.head(10)

## Data Overview

### Keseimbangan Data
Pada bagian ini data diperiksa apakah jumlah data asli dan palsu seimbang.

In [ ]:
#pada bgaian ini melihat apakah data seimbang

print(data.groupby(['label'])['label'].count())
sns.countplot(x="label", data=data)
plt.show()

Dari hasil di atas terlihat bahwa data berita palsu sedikit lebih banyak dibanding dengan data berita asli, tetapi tidak menjadi masalah. Sehingga bisa dikatakan bahwa jumlah data seimbang

### Melihat Berapa Banyak Kelas Subject
Selanjutnya, memeriksa berapa banyak kelas subject yang ada pada data dan menampilkannya dalam bentuk bar.

In [ ]:
#melihat berapa banyak kelas yang terdapat dalam data

print(data["subject"].value_counts())
data.groupby(['subject'])['text'].count().plot(kind="bar")
plt.show()

Berdasarkan hasil di atas, dapat dilihat bahwa pada data terdapat 8 jenis subject, di mana subject terbanyak pada data adalah politicsNews

## Data Preprocessing

### Ubah Subject Menjadi Variabel Dummy
Ubah kolom subject menjadi variabel dummy agar dapat digunakan dalam pemodelan atau analisis lebih lanjut.

In [ ]:
#data = pd.get_dummies(data, columns=["subject"]): Baris ini digunakan untuk mengubah fitur "subject" dalam DataFrame data menjadi variabel dummy
#get_dummies() digunakan untuk menghasilkan setiap nilai dalam fitur "subject" akan diubah menjadi kolom terpisah dalam DataFrame, dengan nilai biner (0 atau 1) yang menunjukkan apakah nilai tersebut ada dalam baris data tertentu atau tidak.

data = pd.get_dummies(data, columns=["subject"])
data.head()

### Hapus Kolom Date
Selanjutnya hapus kolom date karena kolom tersebut tidak diperlukan dalam pemrosesan selanjutnya dan tidak memberikan informasi yang relevan.

In [ ]:
#data = data.drop("date", axis=1): Baris ini digunakan untuk menghapus kolom "date" dari DataFrame 'data' dengan menggunakan fungsi drop()
#axis=1 menunjukkan menghapus kolom (axis 1) bukan baris
#data = data.drop("date", axis=1) adalah untuk menghapus kolom "date" dari DataFrame 'data', karena kolom tersebut tidak diperlukan dalam analisis atau pemrosesan selanjutnya.

data = data.drop("date",axis=1)
data.info()

### Natural Language Processing
Pemrosesan bahasa alami. Pada langkah ini akan memproses fitur teks dan label.

Import nltk dan download modul punkt yang berguna untuk tokenisasi data.

In [ ]:
#import nltk digunakan untuk mengimpor pustaka Natural Language Toolkit (NLTK) ke dalam Python
#nltk.download('punkt') digunakan untuk mengunduh data yang diperlukan oleh NLTK untuk tokenisasi kata. Tokenisasi kata adalah proses memecah teks menjadi unit-unit kata yang lebih kecil atau token.
import nltk
nltk.download('punkt')
print("Unduhan Selesai")

Import nltk dan download modul wordnet yang berguna untuk melakukan lemmatisasi kata serta mencari sinonim dan antonim.

In [ ]:
#nltk.download('wordnet') digunakan untuk mengunduh data yang diperlukan oleh NLTK untuk akses ke WordNet. WordNet adalah kamus leksikal bahasa Inggris yang berisi informasi tentang kata-kata, termasuk definisi, sinonim, antonim, dll.
import nltk
nltk.download('wordnet')
print("Unduhan Selesai")

Selanjutnya dilakukan pemrosesan teks dalam kolom text dari data mulai dari pembersihan (cleaning), konversi ke huruf kecil (lowercasing), tokenisasi kata (word tokenization), lemmatisasi, dan penggabungan kembali teks yang telah diproses. Kemudian hasilnya disimpan dalam list new_text.

In [ ]:
#kode ini adalah untuk memproses teks dalam kolom "text" dari DataFrame data dengan langkah-langkah pembersihan (cleaning), konversi ke huruf kecil (lowercasing), tokenisasi kata (word tokenization), lemmatisasi, dan penggabungan kembali teks yang telah diproses. Hasilnya disimpan dalam list new_text, dan new_text[0] mencetak teks yang telah diproses pertama dari list tersebut.
new_text = []
pattern = "[^a-zA-Z]"
lemma = nltk.WordNetLemmatizer()

for txt in data.text:
    
    txt = re.sub(pattern," ",txt) # Cleaning
    txt = txt.lower() # Lowering
    txt = nltk.word_tokenize(txt) # Tokenizing
    txt = [lemma.lemmatize(word) for word in txt] # Lemmatizing
    txt = " ".join(txt)
    new_text.append(txt)
    
    
new_text[0]

Selanjutnya dilakukan pemrosesan teks dalam kolom title dari data mulai dari pembersihan (cleaning), konversi ke huruf kecil (lowercasing), tokenisasi kata (word tokenization), lemmatisasi, dan penggabungan kembali teks yang telah diproses. Kemudian hasilnya disimpan dalam list new_title.

In [ ]:
# kode ini adalah untuk memproses judul (title) dalam kolom "title" dari DataFrame data dengan langkah-langkah pembersihan (cleaning), konversi ke huruf kecil (lowercasing), tokenisasi kata (word tokenization), lemmatisasi, dan penggabungan kembali judul yang telah diproses. Hasilnya disimpan dalam list new_title, dan new_title[0] mencetak judul yang telah diproses pertama dari list tersebut.
new_title = []
for txt in data.title:
    
    txt = re.sub(pattern," ",txt) # Cleaning
    txt = txt.lower() # Lowering
    txt = nltk.word_tokenize(txt) # Tokenizing
    txt = [lemma.lemmatize(word) for word in txt] # Lemmatizing
    txt = " ".join(txt)
    new_title.append(txt)
new_title[0]

Kode ini digunakan untuk menghasilkan matriks representasi vektor untuk teks judul (title) dan teks utama (text) berdasarkan frekuensi kemunculan kata-kata yang telah diproses sebelumnya menggunakan CountVectorizer.

In [ ]:
#countVectorizer untuk mengubah teks judul (title) dan teks utama (text) menjadi representasi vektor berdasarkan frekuensi kemunculan kata-kata.
#kode ini adalah untuk menghasilkan matriks representasi vektor untuk teks judul (title_matrix) dan teks utama (text_matrix) berdasarkan frekuensi kemunculan kata-kata yang telah diproses sebelumnya menggunakan CountVectorizer.
vectorizer_title = CountVectorizer(stop_words="english",max_features=1000)
vectorizer_text = CountVectorizer(stop_words="english",max_features=4000)

title_matrix = vectorizer_title.fit_transform(new_title).toarray() 
text_matrix = vectorizer_text.fit_transform(new_text).toarray()

print("Selesai")

Hapus kolom "title" dan "text" dari DataFrame data serta mencetak informasi ringkas tentang DataFrame data setelah penghapusan.

In [ ]:
#menghapus kolom "title" dan "text" dari DataFrame data dan mencetak informasi ringkas tentang DataFrame data setelah penghapusan.
data.drop(["title","text"],axis=1,inplace=True)
data.info()

Pastikan bahwa pemrosesan dan transformasi teks telah dilakukan dengan benar, serta untuk memahami ukuran data yang akan digunakan dalam analisis atau pemodelan selanjutnya dengan cara melihat dimensi dari DataFrame data, title_matrix, dan text_matrix setelah dilakukan pemrosesan dan transformasi teks menggunakan CountVectorizer.

In [ ]:
#melihat dimensi dari DataFrame 'data', title_matrix, dan text_matrix setelah dilakukan pemrosesan dan transformasi teks menggunakan CountVectorizer. Ini memberikan informasi tentang jumlah baris dan kolom dalam DataFrame 
print(data.shape)
print(title_matrix.shape)
print(text_matrix.shape)

Langkah selanjutnya adalah membuat target variabel y berdasarkan kolom "label" dalam DataFrame data, dan membuat matriks fitur x dengan menggabungkan beberapa komponen seperti kolom-kolom lain dari DataFrame data, title_matrix, dan text_matrix.

In [ ]:
#membuat target variabel y berdasarkan kolom "label" dalam DataFrame 'data', dan membuat matriks fitur x dengan menggabungkan beberapa komponen seperti kolom-kolom lain dari DataFrame 'data',
# Creating Y
y = data.label
# Creating X
x = np.concatenate((np.array(data.drop("label",axis=1)),title_matrix,text_matrix),axis=1)

Setelah itu pastikan data telah disiapkan dengan benar dengan cara melihat dimensi dari matriks fitur dan target variabel.

In [ ]:
#untuk melihat dimensi dari matriks fitur x dan target variabel y
print(x.shape)
print(y.shape)

### Pembagian Data
Jika data sudah diolah, langkah selanjutnya dilakukan pembagian dataset menjadi subset training dan testing, di mana subset testing sebesar 25% dari jumlah keseluruhan dataset.

In [ ]:
#membagi dataset menjadi subset pelatihan dan pengujian menggunakan fungsi train_test_split, dan kemudian mencetak dimensi dari subset pelatihan dan pengujian matriks fitur serta target variabel
from sklearn.model_selection import train_test_split

# Train Test Split
X_train,X_test,Y_train,Y_test = train_test_split(x,np.array(y),test_size=0.25,random_state=1)
print(X_train.shape)
print(X_test.shape)
print(Y_train.shape)
print(Y_test.shape)